# TP2 - Informe tecnico

Sistema de Deteccion y Clasificacion de Razas de Perros - IA 5.2 Computer Vision.

## Equipo
- Alumno 1 : Manuela Almada
- Alumno 2 : Alfredo Sanz

## 1. Explicacion completa del pipeline

El sistema implementa, de forma incremental, el pipeline clasico de un sistema moderno de vision por computadora para razas de perros:

**Embeddings -> Busqueda por similitud -> Clasificacion -> Deteccion -> Pipeline completo.**

- **Etapa 1 (busqueda por similitud).** Se extrae un embedding de la imagen consultada con un modelo pre-entrenado, se buscan en una base vectorial (PostgreSQL + pgvector) los vecinos mas cercanos y se predice la raza por votacion ponderada de los Top-K (con fallback a "unknown" bajo umbral).
- **Etapa 2 (clasificacion supervisada).** Dos modelos entrenados (ResNet18 fine-tuned y una CNN propia) predicen la raza directamente, devolviendo raza + score.
- **Etapa 3 (deteccion + clasificacion).** YOLO detecta los perros de la imagen, se recorta cada deteccion y se la clasifica reutilizando el modelo de Etapa 2; se muestran las cajas con raza y score de confianza.

La integracion corre sobre una arquitectura desacoplada: backend **FastAPI**, frontend **Gradio** y base vectorial **PostgreSQL + pgvector**, todo orquestado con **Docker Compose**. Cada etapa se expone como una solapa de la aplicacion Gradio, y toda la configuracion (modelo activo, umbrales, cantidad de vecinos, paths, parametros de YOLO) se maneja por variables de entorno (`.env`), sin hardcodeo.

## 2. Dataset

Se utiliza el **70 Dog Breeds Image Dataset** (Kaggle), que ya viene dividido en `train/valid/test`. El analisis de distribucion de clases, la cantidad de imagenes por raza, el grafico de balance y la grilla de muestras se documentan en detalle en `etapa2_colab.ipynb` (Seccion 5).

- Numero de clases: **70 razas**.
- Splits: division oficial del dataset en train / valid / test. El mapeo idx->raza es identico en los tres porque `ImageFolder` ordena las carpetas alfabeticamente y los tres comparten los mismos nombres.
- Conjunto independiente de evaluacion: imagenes descargadas de internet, fuera del dataset, usadas para la evaluacion final. **25 imágenes, 5 imágenes en 5 razas tomadas del dataset (beagle, boxer, dalmation, maltese, schnauzer)**

Cantidades por raza, grafico de distribucion y estadisticas de balance: ver `etapa1_colab.ipynb` y `etapa2_colab.ipynb`.

## 3. Preprocesamiento

Se aplica el **mismo preprocesamiento en indexado, consulta e inferencia** para evitar *train/serve skew*. La lectura es con OpenCV (BGR) y se convierte a RGB, manteniendo la convencion de todo el proyecto.

**Pipeline determinista (valid / test / inferencia):** resize a 224x224 + normalizacion ImageNet (mean/std de ImageNet, lo que esperan los pesos pre-entrenados).

**Data augmentation (solo en entrenamiento):**
- *Horizontal Flip* (p=0.5): los perros son simetricos horizontalmente; no se usa flip vertical (no tendria sentido).
- *Rotacion +/-15 grados*: tolerancia a pequenas inclinaciones de camara.
- *Brillo / contraste* (+/-0.2): robustez a condiciones de iluminacion.
- *Blur* (Gaussian o Motion, uno u otro): robustez a desenfoque y movimiento.
- *Ruido gaussiano* moderado: robustez a sensores de baja calidad.

La justificacion completa y las visualizaciones del pipeline estan en `etapa2_colab.ipynb` (Seccion 6).

## 4. Justificacion de los modelos elegidos

**Etapa 1 - embeddings baseline: ResNet18 pre-entrenada en ImageNet.** La decision mas determinante fue usar ResNet18 y **no** ResNet50: la infraestructura de la base vectorial fija `EMBEDDING_DIM=512`, y la penultima capa de ResNet18 produce exactamente 512 features. Es decir, la eleccion vino **dictada por la configuracion del sistema** (compatibilidad con pgvector), no por buscar maxima capacidad.

**Etapa 2 - Modelo A (ResNet18 fine-tuned) y Modelo B (CNN propia).** Ambos exponen la misma interfaz `backbone`/`head` y los metodos `forward`/`embed`, y terminan en un embedding de **512-d**. Esto los hace intercambiables entre si y compatibles con pgvector, de modo que la busqueda por similitud (Etapa 1) tambien puede funcionar con los modelos entrenados. La CNN propia son 5 bloques convolucionales (32->64->128->256->512) + global average pooling.

**Etapa 3 - YOLOv8n.** Se eligio el modelo *nano* de la familia YOLO: pre-entrenado en COCO (que ya incluye la clase 'dog'), no requiere entrenamiento, liviano y rapido. El trade-off es claro: bajo consumo de memoria y baja latencia a cambio de algo de recall en escenas con oclusion fuerte o sujetos muy similares.

**Trade-offs (precision / velocidad / memoria / complejidad).** ResNet18 vs ResNet50: menor capacidad pero compatibilidad 512-d y menor costo. Clasificacion supervisada vs similitud: mayor precision vs no requerir entrenamiento. YOLOv8n vs variantes mayores (s/m): velocidad y tamano vs recall en multitudes.

## 5. Proceso de entrenamiento e hiperparametros

Ambos modelos comparten el mismo bucle de entrenamiento (`_fit`): optimizador **AdamW**, perdida **CrossEntropyLoss**, scheduler **CosineAnnealingLR**, y seleccion del mejor modelo por *val accuracy*. El mapeo idx->raza (`model.classes`) se guarda junto al checkpoint, de modo que en inferencia la clasificacion traduce el indice a la raza sin reconstruir nada desde el dataset.

**Modelo A - ResNet18 fine-tuned** (viene pre-entrenado; hay que preservar ImageNet):
- LR diferencial: backbone `1e-4`, cabeza `1e-3`.
- Warmup: 2 epocas con el backbone congelado (solo se entrena la cabeza), luego fine-tuning completo.
- Epocas: 20. Weight decay: `1e-4`.

**Modelo B - CNN propia** (entrenada desde cero; no hay pesos previos que preservar):
- LR uniforme `1e-3` para toda la red, sin warmup.
- Epocas: 30. Weight decay: `1e-4`.

Las **curvas de entrenamiento** (loss y accuracy, train vs valid) y la comparacion entre ambos modelos estan en `etapa2_colab.ipynb`. YOLO no se entrena (se usa pre-entrenado).

Los checkpoints pueden encontrarse en este recurso compartido:
https://drive.google.com/drive/folders/1x2COD84Y3bpIcz-H4Kj6hyoGECUWciZW?usp=sharing

## 6. Resultados obtenidos

**Etapa 1 - Busqueda por similitud.**
- NDCG@10: 0.9538
- Justificacion: la busqueda funciona muy bien con razas visualmente **distintivas** (Maltese, Pomeranian, Saint Bernard) y peor con razas visualmente **parecidas** entre si (Bulldog, Shiba Inu), donde los vecinos mas cercanos pueden caer en razas similares.
- Análisis en mayor profundidad es `etapa1_colab.ipynb`.

**Etapa 2 - Clasificacion supervisada** (metricas *macro*, sobre el test set):

| Metrica | Modelo A (ResNet18 FT) | Modelo B (CNN propia) |
|---|---|---|
| Accuracy | 0.9600 | 0.7029  |
| Precision (macro) | 0.9627| 0.7166 |
| Recall (macro) | 0.9600 | 0.7029 |
| F1 (macro) | 0.9598 | 0.6924 |

- **Matriz de confusion:** ver `etapa2_colab.ipynb`.
- **Nota metodologica:** se reportan metricas **macro** para ser justos entre las 70 clases. La *specificity* es casi trivial en un problema multiclase de 70 clases (los verdaderos negativos dominan en cada clase), por lo que no es una metrica discriminante; el analisis se centra en accuracy, precision, recall y F1.

## 7. Comparacion entre enfoques

**Similitud (Etapa 1) vs clasificacion supervisada (Etapa 2).** La busqueda por similitud no requiere entrenamiento y es interpretable (muestra los vecinos que justifican la prediccion), pero depende de la calidad del embedding y de la cobertura de la base. La clasificacion supervisada entrega mayor precision y un score directo de raza, a costa de tener que entrenar.

**ResNet18 fine-tuned vs CNN propia.** El fine-tuning supera ampliamente a la CNN entrenada desde cero (f1-score: ~0.96 vs ~0.69). La brecha refleja el valor del *transfer learning*: con un dataset de 70 clases relativamente chico, partir de pesos de ImageNet aporta muchisimo frente a aprender todo desde cero. La CNN propia es mas liviana, pero claramente menos precisa.

## 8. Problemas encontrados y soluciones implementadas

**Etapa 1 - Busqueda por similitud.**
- *Credenciales de Kaggle en Docker:* la API key antigua no funcionaba dentro del contenedor. **Solucion:** usar `kaggle.json` con `KAGGLE_CONFIG_DIR` + volume mount, manteniendo el archivo fuera del repo.
- *Scripts no montados en `docker-compose.yml`:* la carpeta `scripts/` no estaba montada. **Solucion:** montarla al vuelo y ejecutar con `--entrypoint python` + `PYTHONPATH=/app`.
- *Conflicto de permisos root/usuario en `./data`:* archivos escritos por el contenedor (root) chocaban con el usuario local. **Solucion:** `chown` / ejecutar con `--user`.
- *Consistencia del score con el umbral:* se opto por una busqueda agnostica al store (traer los registros y calcular la similitud en codigo) para que la escala del score sea consistente con el umbral, asumiendo el trade-off frente a la busqueda nativa de pgvector.

**Etapa 2 - Clasificacion.**
- *Archivos desactualizados en Colab:* archivos no sincronizados desde GitHub causaban bugs dificiles de diagnosticar. **Criterio:** verificar siempre que el archivo en ejecucion coincida con el ultimo commit.
- *Perdida de checkpoints al reciclarse el runtime de Colab:* los `.pth` viven en el espacio efimero de Colab y se pierden al cerrar la sesion. **Solucion:** descargar ambos `.pth` apenas termina el entrenamiento (o guardarlos en Google Drive montado).

**Etapa 3 - Deteccion + clasificacion.**
- *Descarga de pesos de YOLO desde internet:* `ultralytics` descarga el `.pt` la primera vez si recibe solo un nombre, lo que romperia la evaluacion sin salida a internet. **Solucion:** bundlear `yolov8n.pt` en `models/` y apuntar `YOLO_MODEL` a la ruta local -> reproducible y offline.
- *Caja sobredimensionada en imagenes muy grandes:* a la resolucion de inferencia por defecto (640 px), una foto de ~4000 px daba una unica caja que abarcaba casi toda la imagen. **Solucion:** hacer configurable la resolucion de inferencia (`YOLO_IMGSZ=1280`), lo que ajusta las cajas en imagenes grandes y cargadas.
- *Techo de recall de YOLOv8n:* el modelo nano no detecta perros con oclusion fuerte en clusteres muy apretados (ej. el perro del medio entre dos casi identicos) ni close-ups que llenan todo el cuadro. **Criterio:** se documenta como limitacion asumida del detector liviano; el detector apunta a escenas con contexto, mientras que los recortes apretados de un solo perro son el dominio de la clasificacion (Etapa 2). El umbral de deteccion es configurable (`YOLO_CONF_THRESHOLD`) si se quisiera mas sensibilidad, a costa de mas falsos positivos.

## 9. Modificaciones fuera de las funciones indicadas

- **Soporte de `imgsz` configurable en deteccion (Etapa 3).** Se agrego el parametro `imgsz` al constructor de `DetectionService` (default 1280), el campo `yolo_imgsz` en `config.py`, su paso en `bootstrap.py` y la variable `YOLO_IMGSZ` en los `.env`. *Justificacion:* robustez de la deteccion en imagenes grandes/cargadas, totalmente configurable por entorno (sin hardcodeo).
- **`YOLO_MODEL` apuntando a ruta local + pesos bundleados.** Se cambio `YOLO_MODEL` a `models/yolov8n.pt` (y `../models/yolov8n.pt` en local) y se incluyo el peso en el repo. *Justificacion:* evaluacion reproducible sin depender de descargas por internet.
- **Extension: captura por webcam en Gradio (Etapa 1).** Se habilito `sources=["upload", "webcam"]` para consultar con la camara del navegador. *Justificacion:* mejora de usabilidad.

- Detección de *Data Leakage* : Durante una etapa posterior del desarrollo se detectó la presencia de *data leakage* entre los conjuntos de entrenamiento, validación y prueba. Mediante el cálculo de hashes MD5 se comprobó que existían imágenes idénticas compartidas entre los distintos subconjuntos del dataset. La presencia de estas imágenes viola la independencia que debe existir entre los datos de entrenamiento y evaluación, pudiendo generar una sobreestimación del rendimiento del modelo al evaluar muestras que ya fueron vistas durante el entrenamiento. Como consecuencia de este análisis se construyó una nueva versión del dataset (`dogs_clean.csv`), eliminando todas las imágenes duplicadas entre los distintos conjuntos. Debido a que este hallazgo se produjo una vez finalizados los experimentos principales, no fue posible repetir íntegramente el trabajo práctico utilizando el nuevo dataset. Sin embargo, la metodología desarrollada permite reproducir todos los experimentos sobre una base de datos libre de *data leakage*, obteniendo una evaluación más representativa de la capacidad de generalización de los modelos.


## **CONSIDERACIONES PARA EJECUTAR CORRECTAMENTE EL SERVICIO (ETAPA 1)**
- Debe descargar el archivo embeddings_dump.sql y tenerlo en la carpeta de su proyecto: [Embeddings](https://drive.google.com/drive/folders/1Puvsj0FIL2ZpimETmsCc208WBmMk5Vz1?usp=drive_link)

- Tener en su carpeta data/dataset los conjuntos de datos y el csv de kaggle:
[Dataset 70 razas de perros](https://www.kaggle.com/datasets/gpiosenka/70-dog-breedsimage-data-set?resource=download)

*Ejecutar los siguientes comandos en orden:*

Borrar en caso de que la tabla ya exista:

docker compose exec -T postgres psql -U dogs_user -d dogs -c "DROP TABLE IF EXISTS embeddings CASCADE; DROP TABLE IF EXISTS perro_embeddings CASCADE;"

Generar la tabla que se conecta con la app:

docker compose exec -T postgres psql -U dogs_user -d dogs -f /tmp/embeddings_dump.sql

Contar la cantidad de elementos de la tabla para verificar que se haya cargado bien:

docker compose exec -T postgres psql -U dogs_user -d dogs -c "SELECT COUNT(*) FROM embeddings;"

